# 01 · Data download (Phase 1)

**Goal:** one frozen raw snapshot of every source.

| Source | File | How |
|---|---|---|
| LTA *COE Bidding Results / Prices* (data.gov.sg) | `data/raw/coe_bidding_results.csv` | API, code below |
| Policy events (LTA / MOT / MAS pages) | `data/raw/policy_events.csv` | entered by hand, one row per event with its source link |

**Frozen cut-off:** the snapshot ends with the **2nd round of October 2026** (1,990 rows, 398 rounds × 5 categories). The download only runs if the file is missing, so every result uses the same data. Set `FORCE = True` only if the group deliberately agrees to move the cut-off.

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import *
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [2]:
from src.data import download_coe, load_raw, check_raw
FORCE = False
raw_text = download_coe(force=FORCE)
print(len(raw_text), "rows")
raw_text.head()

Using frozen snapshot coe_bidding_results.csv (pass force=True to re-download)
1990 rows


,month,bidding_no,vehicle_class,quota,bids_success,bids_received,premium
0,2010-01,1,Category A,1152,1145,1342,18502
1,2010-01,1,Category B,687,679,883,19190
2,2010-01,1,Category C,173,173,265,19001
3,2010-01,1,Category D,373,365,509,889
4,2010-01,1,Category E,586,567,1011,19889


## Checks before moving on

In [3]:
raw = load_raw()
checks = check_raw(raw)
for k, v in checks.items():
    print(f"{k:28s} {v}")

assert checks["equal_rows"], "categories have different row counts"
assert checks["duplicates"] == 0, "duplicate month + round + category"
assert checks["missing_months"] == ["2020-04", "2020-05", "2020-06"], "unexpected gap"
assert checks["months_without_2_rounds"] == []
assert checks["last_month"] == CUTOFF_MONTH
print("\nAll Phase 1 checks passed. Cut-off:", CUTOFF_MONTH, "round", CUTOFF_ROUND)

rows                         1990
rows_per_category            {'Category A': 398, 'Category B': 398, 'Category C': 398, 'Category D': 398, 'Category E': 398}
equal_rows                   True
duplicates                   0
missing_months               ['2020-04', '2020-05', '2020-06']
months_without_2_rounds      []
first_month                  2010-01
last_month                   2026-10

All Phase 1 checks passed. Cut-off: 2026-10 round 2


In [4]:
# values arrive as text, and some contain thousands separators ("1,308"); load_raw strips them
print(raw_text["bids_received"][raw_text["bids_received"].str.contains(",")].head(3).tolist())
raw[["quota", "bids_success", "bids_received", "premium"]].describe()

['1,438', '1,039', '1,028']


,quota,bids_success,bids_received,premium
count,"1,990.000","1,990.000","1,990.000","1,990.000"
mean,571.649,562.495,848.628,"53,769.851"
std,417.864,416.071,626.819,"35,238.011"
min,43.000,39.000,65.000,852.000
25%,290.250,282.000,433.000,"29,902.000"
50%,445.500,433.500,631.500,"50,514.000"
75%,692.250,682.000,"1,067.750","76,610.750"
max,"2,272.000","2,246.000","4,545.000","158,004.000"


## Policy events

Built by hand. Rules: `date` = effective date, `announced` = date it became public (used so that event features never look ahead). Where the announcement day could not be verified, `announced` is set to the effective date (or the end of the announcement month), which can only make the feature *less* informative, never leak.

`direction` = the expected effect on the premium **once the event takes effect**: `down` (more supply or less demand: COE injections, loan curbs, rebate cuts), `up` (less supply or more demand: zero vehicle growth, loan easing, new rebates) or `mixed` (category redefinitions, VES, the suspension). The single `event_next` flag mixes events with opposite effects, so the regression and M4 use the two directional flags instead.

In [5]:
from src.features import load_policy_events
events = load_policy_events()
events[["date", "announced", "type", "direction", "categories", "description"]]

,date,announced,type,direction,categories,description
0,2013-02-26,2013-02-25,financing_rule,down,ALL,MAS caps motor vehicle loans: max LTV 60% (OMV...
1,2014-02-01,2013-09-09,category_redefinition,mixed,A;B,Cat A adds a 97kW (130bhp) power cap on top of...
2,2016-05-27,2016-05-26,financing_rule,up,ALL,MAS eases motor vehicle loans: max LTV 70% / 6...
3,2018-01-01,2017-03-31,rebate_change,mixed,ALL,Vehicular Emissions Scheme (VES) replaces CEVS...
4,2018-02-01,2017-10-23,quota_change,up,ALL,Car vehicle growth rate cut from 0.25% to 0% a...
5,2020-04-07,2020-04-05,suspension,mixed,ALL,COE bidding suspended during the COVID-19 circ...
6,2020-07-06,2020-06-18,suspension,mixed,ALL,COE bidding resumes with the 1st exercise of J...
7,2020-07-06,2020-06-18,injection,down,ALL,"19,490 COEs accumulated during the suspension ..."
8,2021-01-01,2020-11-12,rebate_change,up,ALL,EV Early Adoption Incentive starts; VES Band A...
9,2022-05-04,2022-03-08,category_redefinition,mixed,A;B,Cat A power threshold for fully electric cars ...


In [6]:
print(events["source_note"].str.split(" ").str[0].value_counts().to_string(), "\n")
print(events["direction"].value_counts().to_string())

source_note
primary      13
secondary     1 

direction
down     6
mixed    5
up       3
